# GBD 2023 Starter Analysis

This notebook starts from the reusable starter datasets built from the official 1990-2023 GBD files.
It covers a quick mortality trend view and a global high-BMI exposure profile.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
ROOT = Path('/Users/apple/Documents/lancet-research-platform')
MORTALITY_PATH = Path('/Users/apple/Documents/lancet-research-platform/data/silver/gbd/gbd2023_mortality_s7_both_sex_long.csv')
ADULT_BMI_PATH = Path('/Users/apple/Documents/lancet-research-platform/data/silver/gbd/gbd2023_high_bmi_global_adult_bmi_long.csv')
PREVALENCE_PATH = Path('/Users/apple/Documents/lancet-research-platform/data/silver/gbd/gbd2023_high_bmi_prevalence_locations_long.csv')

mortality = pd.read_csv(MORTALITY_PATH)
adult_bmi = pd.read_csv(ADULT_BMI_PATH)
prevalence = pd.read_csv(PREVALENCE_PATH)

print('mortality shape:', mortality.shape)
print('adult_bmi shape:', adult_bmi.shape)
print('prevalence shape:', prevalence.shape)
mortality.head()

In [ ]:
global_all_cause = mortality.loc[
    (mortality['location_name'] == 'Global')
    & (mortality['cause_name'] == 'All causes')
    & (mortality['metric'] == 'age_standardized_mortality_rate')
].copy()
global_all_cause = global_all_cause.sort_values('year_id')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(global_all_cause['year_id'], global_all_cause['estimate'], marker='o', linewidth=2)
ax.fill_between(global_all_cause['year_id'], global_all_cause['lower'], global_all_cause['upper'], alpha=0.2)
ax.set_title('Global all-cause age-standardized mortality rate')
ax.set_xlabel('Year')
ax.set_ylabel('ASMR per 100,000')
plt.tight_layout()
plt.show()

global_all_cause[['year_id', 'estimate', 'lower', 'upper']]

In [ ]:
top_causes_2023 = mortality.loc[
    (mortality['location_name'] == 'Global')
    & (mortality['metric'] == 'age_standardized_mortality_rate')
    & (mortality['year_id'] == 2023)
    & (mortality['cause_name'] != 'All causes')
].copy()
top_causes_2023 = top_causes_2023.nlargest(15, 'estimate').sort_values('estimate')

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_causes_2023['cause_name'], top_causes_2023['estimate'], color='#b85c38')
ax.set_title('Top 15 global causes by 2023 ASMR')
ax.set_xlabel('ASMR per 100,000')
ax.set_ylabel('Cause')
plt.tight_layout()
plt.show()

top_causes_2023[['cause_name', 'estimate', 'lower', 'upper']].tail(15)

In [ ]:
AGE_GROUP = '20 to 24'
adult_subset = adult_bmi.copy()
if AGE_GROUP not in set(adult_subset['age_group_name']):
    AGE_GROUP = sorted(adult_subset['age_group_name'].dropna().unique())[0]
adult_subset = adult_subset.loc[adult_subset['age_group_name'] == AGE_GROUP].sort_values(['sex', 'year_id'])

fig, ax = plt.subplots(figsize=(8, 4))
for sex, frame in adult_subset.groupby('sex'):
    ax.plot(frame['year_id'], frame['mean'], label=sex, linewidth=2)
ax.set_title(f'Global adult mean BMI ({AGE_GROUP})')
ax.set_xlabel('Year')
ax.set_ylabel('BMI')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

adult_subset[['sex', 'year_id', 'mean', 'lower', 'upper']].head()

In [ ]:
AGE_GROUP = '20 to 24'
LOCATION = 'China' if 'China' in set(prevalence['location_name']) else sorted(prevalence['location_name'].dropna().unique())[0]
indicator_subset = prevalence.loc[
    (prevalence['location_name'] == LOCATION)
    & (prevalence['risk_indicator'].isin([
        'prevalence_of_obesity',
        'prevalence_of_overweight',
        'prevalence_of_obesity_and_overweight',
    ]))
].copy()
if AGE_GROUP not in set(indicator_subset['age_group_name']):
    AGE_GROUP = sorted(indicator_subset['age_group_name'].dropna().unique())[0]
indicator_subset = indicator_subset.loc[indicator_subset['age_group_name'] == AGE_GROUP]
indicator_subset = indicator_subset.sort_values(['risk_indicator', 'year_id'])

fig, ax = plt.subplots(figsize=(8, 4))
for indicator, frame in indicator_subset.groupby('risk_indicator'):
    ax.plot(frame['year_id'], frame['mean'], label=indicator.replace('_', ' '), linewidth=2)
ax.set_title(f'{LOCATION} high-BMI prevalence trends ({AGE_GROUP}, both sexes)')
ax.set_xlabel('Year')
ax.set_ylabel('Prevalence')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

indicator_subset[['location_name', 'risk_indicator', 'year_id', 'mean', 'lower', 'upper']].head()